# Class 5: Missing data and advanced `groupby` in pandas

Today we move from *producing a grouped table* to *building reusable business analysis*. We will audit and handle missing values, then practice named aggregations, multi-column groups, group-level calculations with `transform`, within-group ranking, and reshaping. Finally, you will transfer those skills to a new restaurant dataset.

## Learning objectives

By the end of class, you should be able to:

- detect and summarize missing values with `isna()` and `notna()`;
- choose deliberately between retaining, dropping, and filling missing values;
- create readable summary tables with named aggregations;
- group by more than one category and reshape the result;
- use `transform` to compare each row with its group;
- rank observations within groups and remove groups that are too small;
- turn an exploratory analysis into a short, evidence-based business recommendation.


## Class plan (75 minutes)

| Time | Topic |
|---|---|
| 0–7 min | Review: data quality and split–apply–combine |
| 7–18 min | Finding and handling missing values |
| 18–29 min | Named aggregations |
| 29–39 min | Multiple grouping columns and reshaping |
| 39–49 min | `transform`, within-group ranking, and filtering |
| 49–70 min | Restaurant data-cleaning and exploration activity |
| 70–75 min | Share-out and assessment setup |


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


## 1. Load the bank-marketing data

This dataset records a bank's direct-marketing contacts. The file uses semicolons rather than commas, so we pass `sep=";"` to `read_csv`.


In [ ]:
BANK_URL = "https://raw.githubusercontent.com/jfkoehler/bootcamp_fa_26/main/data/uci_bank_marketing_sample.csv"

bank = pd.read_csv(BANK_URL, sep=";")
bank = bank.assign(
    subscribed=bank["y"].eq("yes"),
    previously_contacted=bank["pdays"].ne(-1),
)

print(bank.shape)
bank.head()


## 2. Missing values and coded missingness

Pandas usually represents a missing value as `NaN`, `None`, or `pd.NA`. The expression `data.isna()` returns `True` wherever pandas recognizes a value as missing. Combining it with `.sum()` counts missing values by column; combining it with `.mean()` gives the proportion missing.


In [ ]:
bank.isna()

In [ ]:
bank.isna().sum()

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": bank.isna().sum(),
    "missing_percent": bank.isna().mean().mul(100),
})

missing_report.sort_values("missing_count", ascending=False)


### A data-quality trap: sentinel values

The bank data appears to have no missing values, but `pdays == -1` means that the client was not previously contacted. Because `-1` is an ordinary number to pandas, `.isna()` does not count it. A data dictionary is essential: missingness can be encoded rather than explicit.

We can use `.mask()` to replace that sentinel with a recognized missing value. Then `.notna()` selects rows where the value is present.


In [ ]:
bank_with_na = bank.assign(
    days_since_previous=lambda frame: frame["pdays"].mask(frame["pdays"].eq(-1))
)




In [ ]:
print("Coded as -1:", bank["pdays"].eq(-1).sum())
print("Recognized as missing:", bank_with_na["days_since_previous"].isna().sum())

In [ ]:
previously_contacted = bank_with_na.loc[
    bank_with_na["days_since_previous"].notna(),
    ["job", "days_since_previous", "previous", "poutcome"],
]

In [ ]:
previously_contacted.head()

### Common missing-value tools

| Goal | Typical pandas expression |
|---|---|
| Find missing cells | `df.isna()` |
| Count missing values by column | `df.isna().sum()` |
| Calculate percent missing | `df.isna().mean() * 100` |
| Keep rows where a value is present | `df[df["column"].notna()]` |
| Display rows missing any selected field | `df[df[columns].isna().any(axis=1)]` |
| Remove rows missing required fields | `df.dropna(subset=columns)` |
| Replace missing values | `df["column"].fillna(value)` |

Neither dropping nor filling is automatically correct. The choice depends on why the data is missing and how the field will be used.


## 3. The `groupby` mental model

`groupby` follows a **split → apply → combine** pattern:

1. **Split** rows into groups using one or more columns.
2. **Apply** a calculation to each group.
3. **Combine** the results into a Series or DataFrame.

The important analytical choice is not the syntax—it is deciding what one row of the result should represent.

**Quick check:** In a summary grouped by `job`, what does one row represent? What changes if we group by both `job` and `housing`?


## 4. Named aggregations

Named aggregation lets us calculate several metrics and give each output a useful name. The pattern is:

```python
new_column=("source_column", "aggregation")
```

The result below gives decision-makers the group size, response rate, and customer profile in one table.


In [ ]:
job_summary = (
    bank.groupby("job", as_index=False)
        .agg(
            customers=("age", "size"),
            subscription_rate=("subscribed", "mean"),
            average_balance=("balance", "mean"),
            median_age=("age", "median"),
        )
        .sort_values("subscription_rate", ascending=False)
)

job_summary


### Checkpoint

- Why use `size` rather than only comparing subscription rates?
- Which job category appears promising?
- What would you want to know before recommending that the bank target it?


## 5. Multiple grouping columns

Grouping by two columns allows us to compare intersections of categories. Here, each row represents one `job`–`housing` combination.


In [ ]:
job_housing = (
    bank.groupby(["job", "housing"], as_index=False)
        .agg(
            customers=("age", "size"),
            subscription_rate=("subscribed", "mean"),
            average_balance=("balance", "mean"),
        )
        .sort_values(["job", "housing"])
)

job_housing.head(12)


### Reshape a grouped result

`unstack` moves one index level into columns. It is useful when a wide comparison is easier to scan than a long table.


In [ ]:
housing_rate_table = (
    bank.groupby(["job", "housing"])["subscribed"]
        .mean()
        .unstack("housing")
        .rename(columns={"no": "no_housing_loan", "yes": "has_housing_loan"})
)

housing_rate_table.assign(
    rate_difference=lambda frame: (
        frame["no_housing_loan"] - frame["has_housing_loan"]
    )
).sort_values("rate_difference", ascending=False)


## 6. `transform`: put group statistics back on every row

An aggregation reduces many rows to one row per group. `transform` returns one value for every original row, so we can compare an individual customer with the customer's group.


In [ ]:
bank_with_context = bank.assign(
    job_average_balance=lambda frame: (
        frame.groupby("job")["balance"].transform("mean")
    ),
    job_subscription_rate=lambda frame: (
        frame.groupby("job")["subscribed"].transform("mean")
    ),
)

In [ ]:
bank_with_context = bank_with_context.assign(
    balance_vs_job_average=lambda frame: (
        frame["balance"] - frame["job_average_balance"]
    )
)

In [ ]:
bank_with_context[
    ["job", "balance", "job_average_balance", "balance_vs_job_average", "job_subscription_rate"]
].head(10)

### Why not use `agg` here?

We still need one row per customer. `transform` preserves the original shape and index; `agg` would collapse the data to one row per job.

**Turn and talk:** Name another group statistic that would be useful on every customer row.


## 7. Rank within a group

`groupby(...).rank()` answers questions such as “Which customers have the largest balances *within each job category*?”


In [ ]:
ranked_customers = bank.assign(
    balance_rank_within_job=(
        bank.groupby("job")["balance"]
            .rank(method="dense", ascending=False)
    )
)

In [ ]:
top_balances_by_job = (
    ranked_customers.query("balance_rank_within_job <= 3")
        .sort_values(["job", "balance_rank_within_job"])
)

In [ ]:
top_balances_by_job[["job", "balance", "balance_rank_within_job"]].head(15)

## 8. Filter entire groups

Small groups can produce unstable rates. `groupby().filter(...)` keeps or removes *whole groups* based on a condition.


In [ ]:
jobs_with_enough_data = (
    bank.groupby("job")
        .filter(lambda group: len(group) >= 200)
)
jobs_with_enough_data.head()

In [ ]:
stable_job_summary = (
    jobs_with_enough_data.groupby("job", as_index=False)
        .agg(
            customers=("age", "size"),
            subscription_rate=("subscribed", "mean"),
        )
        .sort_values("subscription_rate", ascending=False)
)

In [ ]:
stable_job_summary

## 9. Transfer activity: restaurant operations

Now we switch to a new dataset containing 244 restaurant checks. Imagine that you are advising a restaurant manager who wants to understand demand, staffing needs, and the customer experience.

### Data dictionary

| Column | Meaning |
|---|---|
| `total_bill` | Check total in dollars |
| `tip` | Tip amount in dollars |
| `sex` | Recorded category in the source data |
| `smoker` | Whether the party used the smoking section |
| `day` | Day of the week |
| `time` | Lunch or dinner |
| `size` | Party size |

This is a small observational dataset. Treat patterns as clues, not proof of cause. The demographic categories are also limited and should not be used to stereotype customers or workers.


In [ ]:
TIPS_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"

tips_raw = pd.read_csv(TIPS_URL)

print(tips_raw.shape)
tips_raw.head()


### Missing-data lab

The source file is complete, so the next cell creates a **deliberately incomplete practice copy**. This simulates common data-entry and export problems without pretending the original observations were missing.


In [ ]:
tips_with_missing = tips_raw.copy()

tips_with_missing.loc[[4, 26, 98], "tip"] = pd.NA
tips_with_missing.loc[[15, 67], "total_bill"] = pd.NA
tips_with_missing.loc[[29, 121], "time"] = pd.NA

tips_with_missing.loc[
    tips_with_missing[["total_bill", "tip", "time"]].isna().any(axis=1)
]


#### Audit the missingness

The first result counts missing values. The second calculates the percentage of rows missing in each column.


In [ ]:
restaurant_missing_report = pd.DataFrame({
    "missing_count": tips_with_missing.isna().sum(),
    "missing_percent": tips_with_missing.isna().mean().mul(100),
})

restaurant_missing_report.query("missing_count > 0")


#### Make and document cleaning decisions

For this exercise:

- `total_bill` and `time` are required for the operations analysis, so rows missing either field are removed;
- missing tips are flagged, then filled with the median tip for the same day and meal period;
- the original `tips_with_missing` DataFrame is preserved so the cleaning remains auditable.

Median imputation is only a classroom example. It changes the observed data and can reduce variation, so it must be disclosed in a real analysis.


In [ ]:
tips = (
    tips_with_missing
        .assign(tip_was_missing=lambda frame: frame["tip"].isna())
        .dropna(subset=["total_bill", "time"])
        .copy()
)

tips["tip"] = (
    tips.groupby(["day", "time"])["tip"]
        .transform(lambda values: values.fillna(values.median()))
)

print("Rows before cleaning:", len(tips_with_missing))
print("Rows after cleaning:", len(tips))
print("Remaining missing values:", tips.isna().sum().sum())
tips.loc[tips["tip_was_missing"], ["day", "time", "total_bill", "tip"]]


### Cleaning checkpoint

- Which rows did we drop, and why?
- Why preserve `tip_was_missing` after filling the values?
- When might dropping incomplete rows bias the result?
- What alternative to median imputation could you defend?


### Set up two useful measures

- `tip_rate` makes tips comparable across bills of different sizes.
- `bill_per_person` makes checks comparable across party sizes.


In [ ]:
tips = tips.assign(
    tip_rate=lambda frame: frame["tip"] / frame["total_bill"],
    bill_per_person=lambda frame: frame["total_bill"] / frame["size"],
)

tips[["total_bill", "tip", "size", "tip_rate", "bill_per_person"]].head()


## Student exploration (18 minutes)

Work in pairs. Your deliverable is a short recommendation to the restaurant manager supported by at least **two grouped results**.

### A. Understand the data

Check data types, missing values, unique values in the categorical columns, and the range of each numeric measure. Write down one limitation or question about the data.


In [ ]:
# Your code for Part A


### B. Build an operations summary

Create one row for each `day`–`time` combination. Include:

- number of checks;
- total revenue;
- average bill;
- average tip rate;
- average party size.

Sort the result so the highest-revenue sessions appear first. Which sessions appear busiest? Does “highest total revenue” tell the same story as “highest average bill”?


In [ ]:
# Your code for Part B


### C. Compare each check with its session

Use `transform` to add the average `tip_rate` for each `day`–`time` group to every row. Then calculate how far each check's tip rate is above or below its session average.

What does this row-level comparison reveal that the summary in Part B does not?


In [ ]:
# Your code for Part C


### D. Rank checks within sessions

Rank `total_bill` from largest to smallest inside each `day`–`time` group. Display the top three checks in every session.

Are the largest checks also the checks with the highest tip rates?


In [ ]:
# Your code for Part D


### E. Choose an additional segmentation

Choose one useful grouping question. Possibilities include:

- comparing smoking-section and non-smoking-section checks;
- comparing party-size bands that you create with `pd.cut`;
- comparing day, time, and party size together.

Include the group size in your result. Exclude or clearly flag groups with fewer than 10 checks. Explain why your grouping helps answer an operational question.


In [ ]:
# Your code for Part E


### F. Manager memo

Write 100–150 words that includes:

1. one specific operational recommendation;
2. at least two numbers from your analysis;
3. one limitation of the dataset;
4. one additional piece of data you would collect before acting.

> **Recommendation:** Write your memo here.


## Share-out

Be ready to show one table and explain:

- what one row represents;
- how you created the groups;
- which metric matters most for your recommendation;
- why the evidence does—or does not—support action.


## Exit Ticket

Please complete the exit ticket [here](https://docs.google.com/forms/d/e/1FAIpQLSchg71yF9YtePSXQzecPa5zzXOt-npZ2jaEIQcq1Z6NhKPp6w/viewform?usp=sharing&ouid=106596295392954890759).
